# 🛒 Smart Canteen System - Recommender & Customer Segmentation

### Objectives:
1. Perform **Market Basket Analysis** to discover high-affinity item combinations.
2. Measure Association Rules using **Support**, **Confidence**, and **Lift**.
3. Segment customers using **RFM Analysis** and **K-Means Clustering** to drive targeted loyalty promotions.


In [ ]:
import os
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)


## Part 1: Market Basket Analysis & Combo Mining


In [ ]:
data_path = os.path.join('..', 'data', 'canteen_orders.csv')
df = pd.read_csv(data_path)

# Group items by order_id
baskets = df.groupby('order_id')['item_name'].apply(lambda x: sorted(list(set(x))))
total_baskets = len(baskets)

# Frequency tables
item_freq = defaultdict(int)
pair_freq = defaultdict(int)

for b in baskets:
    for item in b:
        item_freq[item] += 1
    for i in range(len(b)):
        for j in range(i + 1, len(b)):
            pair = tuple(sorted([b[i], b[j]]))
            pair_freq[pair] += 1

print(f'Total Baskets Processed: {total_baskets}')
print(f'Unique Item Pairs Found: {len(pair_freq)}')


In [ ]:
# Compute Association Metrics (Support, Confidence, Lift)
rules = []
for (item_a, item_b), count in pair_freq.items():
    support = count / total_baskets
    conf_a_b = count / item_freq[item_a]
    conf_b_a = count / item_freq[item_b]
    lift = (count * total_baskets) / (item_freq[item_a] * item_freq[item_b])
    
    rules.append({
        'Item A': item_a,
        'Item B': item_b,
        'Co-purchases': count,
        'Support (%)': round(support * 100, 2),
        'Conf (A->B) %': round(conf_a_b * 100, 1),
        'Conf (B->A) %': round(conf_b_a * 100, 1),
        'Lift': round(lift, 2)
    })

rules_df = pd.DataFrame(rules).sort_values(by='Co-purchases', ascending=False)
rules_df.head(10)


In [ ]:
# Visualizing Top Canteen Combo Pairs
top_combos = rules_df.head(6).copy()
top_combos['Combo'] = top_combos['Item A'] + ' + ' + top_combos['Item B']

plt.figure(figsize=(10, 4.5))
sns.barplot(data=top_combos, x='Co-purchases', y='Combo', palette='viridis')
plt.title('Top 6 Frequently Co-Ordered Menu Combos', fontsize=13, fontweight='bold')
plt.xlabel('Number of Orders with Both Items')
plt.ylabel('Combo Pair')
plt.show()


## Part 2: Customer Segmentation (K-Means Clustering on RFM)
- **Recency (R)**: Days since customer's last order.
- **Frequency (F)**: Total number of orders placed.
- **Monetary (M)**: Total expenditure in canteen.


In [ ]:
df['date'] = pd.to_datetime(df['date'])
max_date = df['date'].max()

rfm = df.groupby('customer_id').agg(
    recency=('date', lambda x: (max_date - x.max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('total_price', 'sum'),
    role=('user_role', 'first')
).reset_index()

# Normalize features
scaler = StandardScaler()
scaled_rfm = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

# Apply K-Means (k=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['cluster'] = kmeans.fit_predict(scaled_rfm)

# Map cluster numbers to meaningful business labels
cluster_means = rfm.groupby('cluster')['monetary'].mean().sort_values()
label_dict = {
    cluster_means.index[0]: 'Occasional Snackers',
    cluster_means.index[1]: 'Regular Diners',
    cluster_means.index[2]: 'VIP High-Spenders'
}
rfm['segment'] = rfm['cluster'].map(label_dict)

# Display Segment Profiles
rfm.groupby('segment').agg(
    customer_count=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_orders=('frequency', 'mean'),
    avg_spend=('monetary', 'mean')
).round(1)


In [ ]:
# Visualize Customer Segments
plt.figure(figsize=(9, 5))
sns.scatterplot(
    data=rfm,
    x='frequency',
    y='monetary',
    hue='segment',
    palette=['#f59e0b', '#3b82f6', '#10b981'],
    s=80,
    alpha=0.85
)
plt.title('Customer Segmentation: Frequency vs Monetary Spend', fontsize=13, fontweight='bold')
plt.xlabel('Order Frequency (Total Orders)')
plt.ylabel('Total Spend (₹)')
plt.legend(title='Customer Segment')
plt.show()


## Strategic Action Plan for Canteen Management:
1. **Combo Promotions**: Introduce official combo meal deals for high-lift pairs (e.g., *Masala Chai + Samosa Combo* at Rs. 35, *Burger + Cold Coffee* at Rs. 95) to increase average basket size.
2. **VIP Loyalty Program**: The top 25% *VIP High-Spenders* contribute a disproportionate amount of revenue. Provide them with cashless digital passes and special discounts.
3. **Win-back Campaign**: For *Occasional Snackers* who haven't ordered in 30+ days, trigger app notifications or afternoon snack discounts.
